# DSA 02: Arrays, Strings & Hash Maps
## The Most Common Interview Topics -- Master These First

**What you'll learn:**
- Array operations and their time complexities
- The Two Pointers technique (sorted arrays, palindromes)
- The Sliding Window technique (subarrays, substrings)
- Hash maps for O(1) lookups and frequency counting
- String manipulation patterns
- 25+ practice problems with full solutions

**Prerequisites:** DSA_00 (Introduction), DSA_01 (Complexity Analysis)
**Estimated Time:** 5-6 hours

---

> Arrays, strings, and hash maps appear in ~60% of coding interview problems.
> Master these three and you're halfway to passing any coding interview.

---

---
# Section 1: Arrays -- The Foundation

## What is an Array?

An array (Python `list`) stores elements in **contiguous memory**, accessible by index.

```
Index:  0    1    2    3    4
Value: [10,  20,  30,  40,  50]
        ^                   ^
     arr[0]              arr[4]
```

## Time Complexities -- Know These Cold

| Operation | Time | Why |
|-----------|------|-----|
| Access by index `arr[i]` | O(1) | Direct memory address |
| Append to end `arr.append(x)` | O(1) amortized | Occasional resize |
| Insert at position `arr.insert(i, x)` | O(n) | Must shift elements right |
| Delete at position `del arr[i]` | O(n) | Must shift elements left |
| Search (unsorted) `x in arr` | O(n) | Must check each element |
| Search (sorted) binary search | O(log n) | Cut in half each step |
| Sort `arr.sort()` | O(n log n) | Timsort |
| Slice `arr[i:j]` | O(j-i) | Creates new array |
| `len(arr)` | O(1) | Stored as attribute |

## Key Insight

Insert/delete at the FRONT is O(n) because everything must shift.
If you need fast front operations, use `collections.deque` instead.

In [ ]:
# Array operations with timing
import time

print("ARRAY OPERATIONS -- Time Complexity Demo")
print("=" * 60)

arr = list(range(1_000_000))

# O(1) operations
start = time.perf_counter()
_ = arr[500_000]
t = time.perf_counter() - start
print(f"  Access arr[500000]:    {t:.8f}s  -- O(1)")

start = time.perf_counter()
arr.append(999)
t = time.perf_counter() - start
print(f"  Append to end:         {t:.8f}s  -- O(1) amortized")

start = time.perf_counter()
_ = len(arr)
t = time.perf_counter() - start
print(f"  len(arr):              {t:.8f}s  -- O(1)")

# O(n) operations
start = time.perf_counter()
arr.insert(0, -1)
t = time.perf_counter() - start
print(f"  Insert at front:       {t:.6f}s  -- O(n) SLOW!")

start = time.perf_counter()
_ = 999999 in arr
t = time.perf_counter() - start
print(f"  Linear search:         {t:.6f}s  -- O(n)")

# Compare: deque for front operations
from collections import deque
d = deque(range(1_000_000))
start = time.perf_counter()
d.appendleft(-1)
t = time.perf_counter() - start
print(f"  deque.appendleft():    {t:.8f}s  -- O(1) FAST!")

print("\n  Takeaway: Use deque when you need fast front insert/remove.")

---
# Section 2: Two Pointers Technique

## What is Two Pointers?

Use two indices (pointers) that move through the array, usually from opposite
ends or at different speeds. Reduces O(n^2) brute force to O(n).

```
Array: [1, 2, 3, 4, 5, 6, 7, 8, 9]
        ^                         ^
       left                     right

Move left right, move right left, until they meet.
```

## When to Use Two Pointers

- Array is **sorted** (or can be sorted)
- Finding a **pair** that satisfies a condition
- **Palindrome** checking
- **Removing duplicates** from sorted array
- **Partitioning** an array (e.g., move zeros to end)

## The Pattern

```python
left, right = 0, len(arr) - 1
while left < right:
    # Check condition with arr[left] and arr[right]
    # Move left forward or right backward
```

In [ ]:
# Two Pointers: Example 1 -- Two Sum in Sorted Array
print("TWO POINTERS -- Pattern 1: Pair Sum in Sorted Array")
print("=" * 60)

def two_sum_sorted(nums, target):
    """
    Find two numbers in a SORTED array that sum to target.
    Return their indices.
    Time: O(n), Space: O(1)
    """
    left, right = 0, len(nums) - 1

    while left < right:
        current_sum = nums[left] + nums[right]

        if current_sum == target:
            return [left, right]
        elif current_sum < target:
            left += 1    # Need bigger sum -> move left forward
        else:
            right -= 1   # Need smaller sum -> move right backward

    return []  # No pair found

# Demo
nums = [1, 2, 3, 4, 6, 8, 9, 14]
target = 10
print(f"  Array: {nums}")
print(f"  Target: {target}")
result = two_sum_sorted(nums, target)
print(f"  Result: indices {result} -> {nums[result[0]]} + {nums[result[1]]} = {target}")

# Walk through the logic
print("\n  How it works:")
print(f"    left=0 ({nums[0]}), right=7 ({nums[7]}): sum=15 > 10 -> move right")
print(f"    left=0 ({nums[0]}), right=6 ({nums[6]}): sum=10 -> FOUND!")

In [ ]:
# Two Pointers: Example 2 -- Valid Palindrome
print("\nTWO POINTERS -- Pattern 2: Palindrome Check")
print("=" * 60)

def is_palindrome(s):
    """
    Check if string is palindrome (ignoring non-alphanumeric, case-insensitive).
    Time: O(n), Space: O(1)
    """
    left, right = 0, len(s) - 1

    while left < right:
        # Skip non-alphanumeric characters
        while left < right and not s[left].isalnum():
            left += 1
        while left < right and not s[right].isalnum():
            right -= 1

        if s[left].lower() != s[right].lower():
            return False

        left += 1
        right -= 1

    return True

tests = [
    ("racecar", True),
    ("A man, a plan, a canal: Panama", True),
    ("hello", False),
    ("Was it a car or a cat I saw?", True),
]
for s, expected in tests:
    result = is_palindrome(s)
    status = "PASS" if result == expected else "FAIL"
    print(f"  [{status}] is_palindrome("{s}") = {result}")

In [ ]:
# Two Pointers: Example 3 -- Remove Duplicates from Sorted Array
print("\nTWO POINTERS -- Pattern 3: Remove Duplicates (in-place)")
print("=" * 60)

def remove_duplicates(nums):
    """
    Remove duplicates in-place from sorted array.
    Return new length. Modifies array in-place.
    Time: O(n), Space: O(1)
    """
    if not nums:
        return 0

    # 'slow' points to the last unique element
    slow = 0

    for fast in range(1, len(nums)):
        if nums[fast] != nums[slow]:
            slow += 1
            nums[slow] = nums[fast]

    return slow + 1  # Length of unique portion

# Demo
nums = [1, 1, 2, 2, 3, 4, 4, 5, 5, 5]
print(f"  Input:  {nums}")
new_len = remove_duplicates(nums)
print(f"  Output: {nums[:new_len]} (length = {new_len})")
print(f"  No extra array needed -- O(1) space!")

In [ ]:
# Two Pointers: Example 4 -- Move Zeros to End
print("\nTWO POINTERS -- Pattern 4: Move Zeros to End")
print("=" * 60)

def move_zeros(nums):
    """
    Move all zeros to the end while maintaining order of non-zeros.
    In-place. Time: O(n), Space: O(1)
    """
    # 'slow' marks where next non-zero should go
    slow = 0

    for fast in range(len(nums)):
        if nums[fast] != 0:
            nums[slow], nums[fast] = nums[fast], nums[slow]
            slow += 1

    return nums

tests = [
    [0, 1, 0, 3, 12],
    [0, 0, 1],
    [1, 2, 3],
]
for nums in tests:
    original = nums[:]
    result = move_zeros(nums)
    print(f"  {original} -> {result}")

---
# Section 3: Sliding Window Technique

## What is Sliding Window?

Maintain a "window" (subarray/substring) that slides through the data.
Instead of recomputing from scratch each time, update incrementally.

```
Array: [2, 1, 5, 1, 3, 2], window size = 3

Window 1: [2, 1, 5]  sum = 8
Window 2:    [1, 5, 1]  sum = 8 - 2 + 1 = 7  (subtract left, add right)
Window 3:       [5, 1, 3]  sum = 7 - 1 + 3 = 9
Window 4:          [1, 3, 2]  sum = 9 - 5 + 2 = 6
```

Without sliding window: O(n * k) -- recompute each window from scratch
With sliding window: O(n) -- just add/subtract one element

## Two Types

1. **Fixed size window**: Size k is given (e.g., "max sum of k consecutive")
2. **Variable size window**: Expands/shrinks based on condition (e.g., "longest substring without repeating")

## The Pattern

```python
# Fixed window:
window_sum = sum(arr[:k])
for i in range(k, len(arr)):
    window_sum += arr[i] - arr[i - k]  # slide: add new, remove old
    max_sum = max(max_sum, window_sum)

# Variable window:
left = 0
for right in range(len(arr)):
    # Expand window by including arr[right]
    while WINDOW_IS_INVALID:
        # Shrink window from left
        left += 1
```

In [ ]:
# Sliding Window: Fixed Size -- Max Sum Subarray of Size K
print("SLIDING WINDOW -- Fixed Size")
print("=" * 60)

def max_sum_subarray(nums, k):
    """
    Find maximum sum of k consecutive elements.
    Time: O(n), Space: O(1)
    """
    if len(nums) < k:
        return 0

    # Compute first window
    window_sum = sum(nums[:k])
    max_sum = window_sum

    # Slide the window: add new element, remove old
    for i in range(k, len(nums)):
        window_sum += nums[i] - nums[i - k]
        max_sum = max(max_sum, window_sum)

    return max_sum

nums = [2, 1, 5, 1, 3, 2]
k = 3
print(f"  Array: {nums}, k={k}")
print(f"  Max sum of {k} consecutive: {max_sum_subarray(nums, k)}")
print(f"  (Window [5, 1, 3] = 9)")

# Another example
nums = [1, 4, 2, 10, 23, 3, 1, 0, 20]
k = 4
print(f"\n  Array: {nums}, k={k}")
print(f"  Max sum of {k} consecutive: {max_sum_subarray(nums, k)}")

In [ ]:
# Sliding Window: Variable Size -- Longest Substring Without Repeating
print("\nSLIDING WINDOW -- Variable Size")
print("=" * 60)

def length_of_longest_substring(s):
    """
    Find length of longest substring without repeating characters.
    Time: O(n), Space: O(min(n, 26)) for the set
    """
    char_set = set()
    left = 0
    max_len = 0

    for right in range(len(s)):
        # If char already in window, shrink from left until it is not
        while s[right] in char_set:
            char_set.remove(s[left])
            left += 1

        # Add current char to window
        char_set.add(s[right])
        max_len = max(max_len, right - left + 1)

    return max_len

tests = [
    ("abcabcbb", 3),   # "abc"
    ("bbbbb", 1),      # "b"
    ("pwwkew", 3),     # "wke"
    ("abcde", 5),      # whole string
    ("", 0),
]
print("  Longest substring without repeating characters:")
for s, expected in tests:
    result = length_of_longest_substring(s)
    status = "PASS" if result == expected else "FAIL"
    print(f"    [{status}] "{s}" -> {result}")

In [ ]:
# Sliding Window: Variable Size -- Minimum Size Subarray Sum
print("\nSLIDING WINDOW -- Minimum Size Subarray Sum")
print("=" * 60)

def min_subarray_len(target, nums):
    """
    Find minimum length subarray with sum >= target.
    Time: O(n), Space: O(1)
    """
    left = 0
    window_sum = 0
    min_len = float("inf")

    for right in range(len(nums)):
        window_sum += nums[right]

        # Shrink window while sum is valid
        while window_sum >= target:
            min_len = min(min_len, right - left + 1)
            window_sum -= nums[left]
            left += 1

    return min_len if min_len != float("inf") else 0

tests = [
    (7, [2, 3, 1, 2, 4, 3], 2),    # [4, 3]
    (4, [1, 4, 4], 1),             # [4]
    (11, [1, 1, 1, 1, 1], 0),     # impossible
]
print("  Minimum length subarray with sum >= target:")
for target, nums, expected in tests:
    result = min_subarray_len(target, nums)
    status = "PASS" if result == expected else "FAIL"
    print(f"    [{status}] target={target}, nums={nums} -> {result}")

---
# Section 4: Hash Maps -- The Swiss Army Knife

## Why Hash Maps Are So Powerful

A hash map (Python `dict`) gives you O(1) average time for:
- **Insert**: `d[key] = value`
- **Lookup**: `d[key]` or `key in d`
- **Delete**: `del d[key]`

This turns many O(n^2) problems into O(n).

## The 4 Core Hash Map Patterns

### Pattern 1: Frequency Counting
```python
freq = {}
for x in arr:
    freq[x] = freq.get(x, 0) + 1
```

### Pattern 2: Complement Lookup (Two Sum)
```python
seen = {}
for i, x in enumerate(arr):
    if target - x in seen:
        return [seen[target - x], i]
    seen[x] = i
```

### Pattern 3: Grouping
```python
groups = defaultdict(list)
for item in items:
    key = compute_key(item)
    groups[key].append(item)
```

### Pattern 4: First/Last Occurrence
```python
first_seen = {}
for i, x in enumerate(arr):
    if x not in first_seen:
        first_seen[x] = i
```

In [ ]:
# Hash Map Pattern 1: Frequency Counting
from collections import Counter, defaultdict

print("HASH MAP PATTERN 1: Frequency Counting")
print("=" * 60)

# Basic counting
def top_k_frequent(nums, k):
    """
    Find k most frequent elements.
    Time: O(n), Space: O(n)
    """
    freq = Counter(nums)
    return [x for x, _ in freq.most_common(k)]

nums = [1, 1, 1, 2, 2, 3, 4, 4, 4, 4]
print(f"  Array: {nums}")
print(f"  Top 2 frequent: {top_k_frequent(nums, 2)}")

# Valid Anagram
def is_anagram(s, t):
    """
    Check if t is an anagram of s.
    Time: O(n), Space: O(1) -- at most 26 lowercase letters
    """
    if len(s) != len(t):
        return False
    return Counter(s) == Counter(t)

print(f"\n  Anagram checks:")
tests = [("anagram", "nagaram", True), ("rat", "car", False), ("listen", "silent", True)]
for s, t, expected in tests:
    result = is_anagram(s, t)
    status = "PASS" if result == expected else "FAIL"
    print(f"    [{status}] is_anagram("{s}", "{t}") = {result}")

In [ ]:
# Hash Map Pattern 2: Two Sum (Complement Lookup)
print("\nHASH MAP PATTERN 2: Complement Lookup")
print("=" * 60)

def two_sum(nums, target):
    """
    Find indices of two numbers that add to target.
    Time: O(n), Space: O(n)
    """
    seen = {}  # value -> index
    for i, num in enumerate(nums):
        complement = target - num
        if complement in seen:
            return [seen[complement], i]
        seen[num] = i
    return []

print("  Two Sum:")
tests = [
    ([2, 7, 11, 15], 9, [0, 1]),
    ([3, 2, 4], 6, [1, 2]),
    ([3, 3], 6, [0, 1]),
]
for nums, target, expected in tests:
    result = two_sum(nums, target)
    status = "PASS" if result == expected else "FAIL"
    print(f"    [{status}] two_sum({nums}, {target}) = {result}")

In [ ]:
# Hash Map Pattern 3: Grouping
print("\nHASH MAP PATTERN 3: Grouping (Group Anagrams)")
print("=" * 60)

def group_anagrams(strs):
    """
    Group strings that are anagrams of each other.
    Time: O(n * k log k) where k = max string length
    Space: O(n * k)
    """
    groups = defaultdict(list)
    for s in strs:
        key = tuple(sorted(s))  # Anagrams have same sorted form
        groups[key].append(s)
    return list(groups.values())

words = ["eat", "tea", "tan", "ate", "nat", "bat"]
print(f"  Input: {words}")
result = group_anagrams(words)
print(f"  Groups: {result}")

# DE connection: This is GROUP BY in Python!
print("\n  DE Connection: This is exactly what SQL GROUP BY does!")
print("  SELECT sorted_chars, collect_list(word) GROUP BY sorted_chars")

In [ ]:
# Hash Map Pattern 4: Longest Consecutive Sequence
print("\nHASH MAP PATTERN 4: Set for O(1) lookups")
print("=" * 60)

def longest_consecutive(nums):
    """
    Find length of longest consecutive sequence.
    Example: [100, 4, 200, 1, 3, 2] -> 4 (sequence: 1,2,3,4)
    Time: O(n), Space: O(n)
    """
    num_set = set(nums)
    max_length = 0

    for num in num_set:
        # Only start counting from the BEGINNING of a sequence
        if num - 1 not in num_set:
            current = num
            length = 1

            while current + 1 in num_set:
                current += 1
                length += 1

            max_length = max(max_length, length)

    return max_length

tests = [
    ([100, 4, 200, 1, 3, 2], 4),
    ([0, 3, 7, 2, 5, 8, 4, 6, 0, 1], 9),
]
for nums, expected in tests:
    result = longest_consecutive(nums)
    status = "PASS" if result == expected else "FAIL"
    print(f"  [{status}] longest_consecutive({nums}) = {result}")

print("\n  Key insight: Only start from numbers that have no predecessor.")
print("  This ensures each element is visited at most twice -> O(n).")

---
# Section 5: String Manipulation Patterns

## Key Facts About Python Strings

- Strings are **immutable** -- you cannot modify them in place
- Every operation that "changes" a string creates a NEW string
- Concatenation in a loop is O(n^2) -- use `"".join(list)` instead

## Common String Operations

| Operation | Time | Notes |
|-----------|------|-------|
| `s[i]` | O(1) | Index access |
| `len(s)` | O(1) | Cached |
| `s + t` | O(n+m) | Creates new string |
| `s in t` (substring) | O(n*m) | Substring search |
| `s.split()` | O(n) | Creates list |
| `"".join(list)` | O(n) | Efficient concatenation |
| `s[::-1]` | O(n) | Reverse |
| `s.lower()`, `s.upper()` | O(n) | Creates new string |

## Critical Anti-Pattern

```python
# SLOW: O(n^2) -- creates new string each iteration
result = ""
for char in chars:
    result += char  # Each += copies entire string!

# FAST: O(n) -- build list, join once
parts = []
for char in chars:
    parts.append(char)
result = "".join(parts)
```

In [ ]:
# String patterns
print("STRING PATTERNS")
print("=" * 60)

# Pattern 1: Reverse words in a string
def reverse_words(s):
    """Reverse order of words. O(n) time."""
    return " ".join(s.split()[::-1])

print("1. Reverse words:")
tests = ["the sky is blue", "  hello world  ", "a good   example"]
for s in tests:
    print(f"   "{s}" -> "{reverse_words(s)}"")

# Pattern 2: Check if one string is a rotation of another
def is_rotation(s1, s2):
    """Check if s2 is a rotation of s1. O(n) time."""
    if len(s1) != len(s2):
        return False
    return s2 in s1 + s1  # Clever trick!

print("\n2. String rotation:")
print(f"   is_rotation('waterbottle', 'erbottlewat') = {is_rotation('waterbottle', 'erbottlewat')}")
print(f"   is_rotation('hello', 'ohell') = {is_rotation('hello', 'ohell')}")

# Pattern 3: String compression
def compress(s):
    """Compress: 'aabcccccaaa' -> 'a2b1c5a3'. O(n) time."""
    if not s:
        return s
    result = []
    count = 1
    for i in range(1, len(s)):
        if s[i] == s[i-1]:
            count += 1
        else:
            result.append(s[i-1] + str(count))
            count = 1
    result.append(s[-1] + str(count))
    compressed = "".join(result)
    return compressed if len(compressed) < len(s) else s

print("\n3. String compression:")
tests = ["aabcccccaaa", "abc", "aabb"]
for s in tests:
    print(f"   "{s}" -> "{compress(s)}"")

---
# Section 6: Classic Interview Problems

These problems come up CONSTANTLY in interviews. Know them well.

In [ ]:
# Classic 1: Best Time to Buy and Sell Stock
print("CLASSIC PROBLEMS")
print("=" * 60)

def max_profit(prices):
    """
    Find max profit from one buy and one sell.
    Time: O(n), Space: O(1)

    DE Connection: This is like finding the maximum gain
    between any two timestamps in a time series!
    """
    if not prices:
        return 0

    min_price = prices[0]
    max_profit_val = 0

    for price in prices[1:]:
        # Track the minimum price seen so far
        min_price = min(min_price, price)
        # Calculate profit if we sell at current price
        profit = price - min_price
        max_profit_val = max(max_profit_val, profit)

    return max_profit_val

print("1. Best Time to Buy and Sell Stock:")
tests = [
    ([7, 1, 5, 3, 6, 4], 5),   # Buy at 1, sell at 6
    ([7, 6, 4, 3, 1], 0),      # Never profitable
]
for prices, expected in tests:
    result = max_profit(prices)
    status = "PASS" if result == expected else "FAIL"
    print(f"   [{status}] prices={prices} -> profit={result}")

In [ ]:
# Classic 2: Product of Array Except Self
def product_except_self(nums):
    """
    Return array where output[i] = product of all elements except nums[i].
    WITHOUT using division. Time: O(n), Space: O(1) extra (output doesn't count).
    """
    n = len(nums)
    result = [1] * n

    # Left pass: result[i] = product of everything LEFT of i
    prefix = 1
    for i in range(n):
        result[i] = prefix
        prefix *= nums[i]

    # Right pass: multiply by product of everything RIGHT of i
    suffix = 1
    for i in range(n - 1, -1, -1):
        result[i] *= suffix
        suffix *= nums[i]

    return result

print("\n2. Product of Array Except Self:")
tests = [
    ([1, 2, 3, 4], [24, 12, 8, 6]),
    ([2, 3, 4, 5], [60, 40, 30, 24]),
    ([-1, 1, 0, -3, 3], [0, 0, 9, 0, 0]),
]
for nums, expected in tests:
    result = product_except_self(nums)
    status = "PASS" if result == expected else "FAIL"
    print(f"   [{status}] {nums} -> {result}")
print("   Key: Prefix product from left * suffix product from right")

In [ ]:
# Classic 3: Container With Most Water
def max_area(height):
    """
    Find two lines that together with x-axis form container with most water.
    Time: O(n), Space: O(1)
    """
    left, right = 0, len(height) - 1
    max_water = 0

    while left < right:
        width = right - left
        h = min(height[left], height[right])
        max_water = max(max_water, width * h)

        # Move the shorter line inward (it limits the height)
        if height[left] < height[right]:
            left += 1
        else:
            right -= 1

    return max_water

print("\n3. Container With Most Water:")
tests = [
    ([1, 8, 6, 2, 5, 4, 8, 3, 7], 49),
    ([1, 1], 1),
]
for height, expected in tests:
    result = max_area(height)
    status = "PASS" if result == expected else "FAIL"
    print(f"   [{status}] height={height} -> area={result}")
print("   Two pointers: always move the shorter side inward.")

In [ ]:
# Classic 4: 3Sum
def three_sum(nums):
    """
    Find all unique triplets that sum to zero.
    Time: O(n^2), Space: O(1) extra (sorting is in-place)
    """
    nums.sort()
    result = []

    for i in range(len(nums) - 2):
        # Skip duplicates for first element
        if i > 0 and nums[i] == nums[i - 1]:
            continue

        # Two pointers for the remaining two elements
        left, right = i + 1, len(nums) - 1
        while left < right:
            total = nums[i] + nums[left] + nums[right]

            if total == 0:
                result.append([nums[i], nums[left], nums[right]])
                # Skip duplicates
                while left < right and nums[left] == nums[left + 1]:
                    left += 1
                while left < right and nums[right] == nums[right - 1]:
                    right -= 1
                left += 1
                right -= 1
            elif total < 0:
                left += 1
            else:
                right -= 1

    return result

print("\n4. 3Sum (find triplets summing to 0):")
nums = [-1, 0, 1, 2, -1, -4]
print(f"   Input: {nums}")
print(f"   Triplets: {three_sum(nums)}")
print("   Pattern: Fix one element, two-pointer for the other two.")

---
# Section 7: Your Turn -- Practice Problems

Solve these yourself before looking at solutions.

In [ ]:
# ============================================
# YOUR TURN: Problem 1 -- Maximum Subarray (Kadane's Algorithm)
# ============================================
# Find the contiguous subarray with the largest sum.
#
# Example: [-2, 1, -3, 4, -1, 2, 1, -5, 4] -> 6 (subarray [4, -1, 2, 1])
#
# Hint: At each position, decide: extend current subarray or start new?
#       current_sum = max(num, current_sum + num)

def max_subarray(nums):
    pass  # Your solution here


In [ ]:
# ============================================
# YOUR TURN: Problem 2 -- Find All Duplicates in an Array
# ============================================
# Given array of n integers where 1 <= a[i] <= n,
# some elements appear twice. Find all duplicates.
#
# Example: [4, 3, 2, 7, 8, 2, 3, 1] -> [2, 3]
#
# Challenge: Can you do it in O(n) time and O(1) extra space?
# Hint: Use the array itself as a hash map (mark visited indices)

def find_duplicates(nums):
    pass  # Your solution here


In [ ]:
# ============================================
# YOUR TURN: Problem 3 -- Longest Palindromic Substring
# ============================================
# Find the longest palindromic substring in s.
#
# Example: "babad" -> "bab" or "aba"
# Example: "cbbd" -> "bb"
#
# Hint: Expand around center. Try each index as center of
#       odd-length palindrome AND even-length palindrome.

def longest_palindrome(s):
    pass  # Your solution here


---
# Solutions

In [ ]:
# SOLUTION 1: Maximum Subarray (Kadane's Algorithm)
def max_subarray(nums):
    """
    Kadane's algorithm. At each position, either:
    - Extend the current subarray (current_sum + num)
    - Start fresh from this element (num)
    Time: O(n), Space: O(1)
    """
    current_sum = nums[0]
    max_sum = nums[0]

    for num in nums[1:]:
        current_sum = max(num, current_sum + num)
        max_sum = max(max_sum, current_sum)

    return max_sum

print("SOLUTION 1: Maximum Subarray (Kadane's)")
tests = [
    ([-2, 1, -3, 4, -1, 2, 1, -5, 4], 6),
    ([1], 1),
    ([5, 4, -1, 7, 8], 23),
    ([-1, -2, -3], -1),
]
for nums, expected in tests:
    result = max_subarray(nums)
    status = "PASS" if result == expected else "FAIL"
    print(f"  [{status}] max_subarray({nums}) = {result}")
print("\n  DE Connection: Finding the best performing time window in metrics!")

In [ ]:
# SOLUTION 2: Find All Duplicates
def find_duplicates(nums):
    """
    Mark indices as visited by negating values.
    If we visit an already-negative index, that number is a duplicate.
    Time: O(n), Space: O(1)
    """
    result = []
    for num in nums:
        idx = abs(num) - 1  # Map value to index (1-indexed -> 0-indexed)
        if nums[idx] < 0:
            result.append(abs(num))
        else:
            nums[idx] = -nums[idx]
    return result

print("\nSOLUTION 2: Find All Duplicates")
tests = [
    ([4, 3, 2, 7, 8, 2, 3, 1], [2, 3]),
    ([1, 1, 2], [1]),
    ([1], []),
]
for nums, expected in tests:
    result = find_duplicates(nums[:])  # Copy to not modify original
    status = "PASS" if sorted(result) == sorted(expected) else "FAIL"
    print(f"  [{status}] find_duplicates({nums}) = {result}")
print("  Trick: Use the array itself as a hash map via index negation!")

In [ ]:
# SOLUTION 3: Longest Palindromic Substring
def longest_palindrome(s):
    """
    Expand around each center. Try both odd and even length palindromes.
    Time: O(n^2), Space: O(1)
    """
    if not s:
        return ""

    def expand(left, right):
        """Expand outward while characters match."""
        while left >= 0 and right < len(s) and s[left] == s[right]:
            left -= 1
            right += 1
        return s[left + 1:right]

    result = ""
    for i in range(len(s)):
        # Odd length palindrome (center at i)
        odd = expand(i, i)
        # Even length palindrome (center between i and i+1)
        even = expand(i, i + 1)

        if len(odd) > len(result):
            result = odd
        if len(even) > len(result):
            result = even

    return result

print("\nSOLUTION 3: Longest Palindromic Substring")
tests = [
    ("babad", ["bab", "aba"]),  # Either is valid
    ("cbbd", ["bb"]),
    ("racecar", ["racecar"]),
]
for s, valid_answers in tests:
    result = longest_palindrome(s)
    status = "PASS" if result in valid_answers else "FAIL"
    print(f"  [{status}] longest_palindrome("{s}") = "{result}"")

---
# Section 8: Summary & Pattern Recognition

## When to Use Each Pattern

| Problem Type | Pattern | Time |
|-------------|---------|------|
| Find pair with sum = target (sorted) | Two Pointers | O(n) |
| Find pair with sum = target (unsorted) | Hash Map | O(n) |
| Max/min sum of k consecutive | Fixed Sliding Window | O(n) |
| Longest substring with constraint | Variable Sliding Window | O(n) |
| Count frequencies | Hash Map (Counter) | O(n) |
| Group items by property | Hash Map (defaultdict) | O(n) |
| Check palindrome | Two Pointers | O(n) |
| Remove duplicates (sorted) | Two Pointers (slow/fast) | O(n) |
| Subarray sum problems | Prefix sum or Sliding Window | O(n) |

## LeetCode Problems to Practice

**Easy:**
- Two Sum (#1)
- Best Time to Buy and Sell Stock (#121)
- Contains Duplicate (#217)
- Valid Anagram (#242)
- Missing Number (#268)

**Medium:**
- 3Sum (#15)
- Container With Most Water (#11)
- Longest Substring Without Repeating Characters (#3)
- Group Anagrams (#49)
- Product of Array Except Self (#238)
- Maximum Subarray (#53)
- Longest Consecutive Sequence (#128)

---
*Next: DSA_03 -- Linked Lists, Stacks & Queues*

In [ ]:
# Final summary
print("KEY TAKEAWAYS FROM DSA_02")
print("=" * 50)
print()
print("1. Arrays: O(1) index access, O(n) insert/delete")
print("2. Two Pointers: O(n) for sorted array pair problems")
print("3. Sliding Window: O(n) for subarray/substring problems")
print("4. Hash Maps: O(1) lookup turns O(n^2) into O(n)")
print("5. Strings: Immutable -- use join() for concatenation")
print()
print("Top interview patterns from this notebook:")
print("  - Two Sum (hash map complement)")
print("  - Sliding Window (longest/shortest with constraint)")
print("  - Two Pointers (sorted arrays, palindromes)")
print("  - Frequency Counting (Counter, anagrams)")
print()
print("Next: DSA_03 -- Linked Lists, Stacks & Queues")